# Binance USD-M Futures Partial Depth Service

This notebook shows how to call the local `BinanceFuturesDepthService` module from Jupyter. It subscribes to Binance USD-M Futures partial book depth streams, keeps the latest in-memory top-N depth snapshot per symbol, and reads bid1/bid2/ask1/ask2 from that table.

Run the cleanup cell at the end when you are done because the depth service starts a background receiver thread.

## 1. Import Local Package

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_path():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/home/suncong/binance_klines_data_fetch"),
    ]
    for candidate in candidates:
        if (candidate / "binance_klines_data_fetch").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not find the binance_klines_data_fetch repo path")


repo_path = find_repo_path()
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

from binance_klines_data_fetch import BinanceDepthConfig, BinanceFuturesDepthService

print("Imported package from:", repo_path)

Imported package from: /home/suncong/binance_klines_data_fetch


## 2. Configure Partial Depth Streams

`levels` must be `5`, `10`, or `20`. `speed_ms` must be `100`, `250`, or `500`; `250` maps to the no-suffix stream name, for example `btcusdt@depth5`.

In [2]:
config = BinanceDepthConfig(
    symbols=["BTCUSDT", "ETHUSDT"],
    levels=5,
    speed_ms=100,
    read_timeout_seconds=30.0,
    startup_timeout_seconds=30.0,
)

service = BinanceFuturesDepthService(config)
print(service.build_url())

wss://fstream.binance.com/public/stream?streams=btcusdt@depth5@100ms/ethusdt@depth5@100ms


## 3. Start The Background Receiver

This opens one real Binance WebSocket connection. `block_until_ready=True` waits until all configured symbols have at least one non-stale snapshot.

In [3]:
service.start(block_until_ready=True, timeout=30.0)
status = service.status()
print("ready:", status.ready)
print("connected:", status.connected)
print("symbols:", status.symbols)

ready: True
connected: True
symbols: ('BTCUSDT', 'ETHUSDT')


## 4. Read Bid1/Bid2/Ask1/Ask2

In [4]:
def top_two_rows(snapshot):
    rows = []
    sample_time_ms = time.time_ns() // 1_000_000
    quote_age_ms = sample_time_ms - snapshot.event_time_ms

    for side_name, levels in [("bid", snapshot.bids[:2]), ("ask", snapshot.asks[:2])]:
        for level in levels:
            rows.append(
                {
                    "symbol": snapshot.symbol,
                    "side": side_name,
                    "level": level.level,
                    "price": level.price,
                    "qty": level.qty,
                    "event_time_ms": snapshot.event_time_ms,
                    "local_recv_time_ms": snapshot.local_recv_time_ms,
                    "quote_age_ms": quote_age_ms,
                    "receive_latency_ms": snapshot.receive_latency_ms,
                    "is_stale": snapshot.is_stale,
                    "sequence_gap": snapshot.sequence_gap,
                    "final_update_id": snapshot.final_update_id,
                }
            )
    return rows


rows = []
for symbol, snapshot in service.get_all_latest().items():
    if snapshot.is_stale or snapshot.sequence_gap:
        print(f"Skipping {symbol}: stale={snapshot.is_stale}, sequence_gap={snapshot.sequence_gap}")
        continue
    if len(snapshot.bids) < 2 or len(snapshot.asks) < 2:
        print(f"Skipping {symbol}: not enough depth levels")
        continue
    rows.extend(top_two_rows(snapshot))

display(pd.DataFrame(rows))

,symbol,side,level,price,qty,event_time_ms,local_recv_time_ms,quote_age_ms,receive_latency_ms,is_stale,sequence_gap,final_update_id
0,BTCUSDT,bid,1,74001.50,1.419,1780208693946,1780208693993,117,47,False,False,10669735434791
1,BTCUSDT,bid,2,74001.40,0.003,1780208693946,1780208693993,117,47,False,False,10669735434791
2,BTCUSDT,ask,1,74001.60,21.269,1780208693946,1780208693993,117,47,False,False,10669735434791
3,BTCUSDT,ask,2,74001.70,0.007,1780208693946,1780208693993,117,47,False,False,10669735434791
4,ETHUSDT,bid,1,2029.39,3.599,1780208693968,1780208694016,95,48,False,False,10669735436548
5,ETHUSDT,bid,2,2029.38,0.014,1780208693968,1780208694016,95,48,False,False,10669735436548
6,ETHUSDT,ask,1,2029.40,472.470,1780208693968,1780208694016,95,48,False,False,10669735436548
7,ETHUSDT,ask,2,2029.41,137.752,1780208693968,1780208694016,95,48,False,False,10669735436548


## 5. Inspect Full Top-N Snapshots

In [5]:
def snapshot_to_frame(snapshot):
    rows = []
    for side_name, levels in [("bid", snapshot.bids), ("ask", snapshot.asks)]:
        for level in levels:
            rows.append(
                {
                    "symbol": snapshot.symbol,
                    "side": side_name,
                    "level": level.level,
                    "price": level.price,
                    "qty": level.qty,
                    "spread": snapshot.spread,
                    "spread_bps": snapshot.spread_bps,
                    "mid_price": snapshot.mid_price,
                    "final_update_id": snapshot.final_update_id,
                }
            )
    return pd.DataFrame(rows)


for symbol in config.symbols:
    snapshot = service.get_latest(symbol)
    if snapshot is None:
        print(symbol, "has no snapshot yet")
        continue
    print(symbol, "stale=", snapshot.is_stale, "sequence_gap=", snapshot.sequence_gap)
    display(snapshot_to_frame(snapshot))

BTCUSDT stale= False sequence_gap= False


,symbol,side,level,price,qty,spread,spread_bps,mid_price,final_update_id
0,BTCUSDT,bid,1,74001.50,12.882,0.10,0.01351323046611861508306244937,74001.55,10669738232717
1,BTCUSDT,bid,2,74001.40,2.690,0.10,0.01351323046611861508306244937,74001.55,10669738232717
2,BTCUSDT,bid,3,74001.30,0.010,0.10,0.01351323046611861508306244937,74001.55,10669738232717
3,BTCUSDT,bid,4,74001.20,0.078,0.10,0.01351323046611861508306244937,74001.55,10669738232717
4,BTCUSDT,bid,5,74001.10,0.002,0.10,0.01351323046611861508306244937,74001.55,10669738232717
5,BTCUSDT,ask,1,74001.60,5.690,0.10,0.01351323046611861508306244937,74001.55,10669738232717
6,BTCUSDT,ask,2,74001.70,0.003,0.10,0.01351323046611861508306244937,74001.55,10669738232717
7,BTCUSDT,ask,3,74001.80,0.004,0.10,0.01351323046611861508306244937,74001.55,10669738232717
8,BTCUSDT,ask,4,74001.90,0.002,0.10,0.01351323046611861508306244937,74001.55,10669738232717
9,BTCUSDT,ask,5,74002.00,0.049,0.10,0.01351323046611861508306244937,74001.55,10669738232717


ETHUSDT stale= False sequence_gap= False


,symbol,side,level,price,qty,spread,spread_bps,mid_price,final_update_id
0,ETHUSDT,bid,1,2029.34,130.302,0.01,0.04927698346018050159041464118,2029.345,10669738234565
1,ETHUSDT,bid,2,2029.33,0.124,0.01,0.04927698346018050159041464118,2029.345,10669738234565
2,ETHUSDT,bid,3,2029.32,5.272,0.01,0.04927698346018050159041464118,2029.345,10669738234565
3,ETHUSDT,bid,4,2029.31,1.589,0.01,0.04927698346018050159041464118,2029.345,10669738234565
4,ETHUSDT,bid,5,2029.30,0.149,0.01,0.04927698346018050159041464118,2029.345,10669738234565
5,ETHUSDT,ask,1,2029.35,167.560,0.01,0.04927698346018050159041464118,2029.345,10669738234565
6,ETHUSDT,ask,2,2029.36,3.059,0.01,0.04927698346018050159041464118,2029.345,10669738234565
7,ETHUSDT,ask,3,2029.37,36.262,0.01,0.04927698346018050159041464118,2029.345,10669738234565
8,ETHUSDT,ask,4,2029.38,0.052,0.01,0.04927698346018050159041464118,2029.345,10669738234565
9,ETHUSDT,ask,5,2029.39,23.212,0.01,0.04927698346018050159041464118,2029.345,10669738234565


## 6. Watch Updates Briefly

This cell samples the in-memory latest table every second for 10 seconds. It does not write CSV, Parquet, or database records.

In [6]:
for _ in range(10):
    sample_time_ms = time.time_ns() // 1_000_000
    rows = []
    for symbol in config.symbols:
        snapshot = service.get_latest(symbol)
        if snapshot is None:
            rows.append({"symbol": symbol, "state": "missing"})
            continue
        rows.append(
            {
                "symbol": symbol,
                "state": "stale" if snapshot.is_stale else "live",
                "sequence_gap": snapshot.sequence_gap,
                "bid1": snapshot.bids[0].price if snapshot.bids else None,
                "ask1": snapshot.asks[0].price if snapshot.asks else None,
                "spread_bps": snapshot.spread_bps,
                "quote_age_ms": sample_time_ms - snapshot.event_time_ms,
                "final_update_id": snapshot.final_update_id,
            }
        )
    display(pd.DataFrame(rows))
    time.sleep(1.0)

,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.50,74001.60,0.01351323046611861508306244937,168,10669739146051
1,ETHUSDT,live,False,2029.16,2029.17,0.04928135464587650585339289806,118,10669739148856


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,136,10669739263122
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,80,10669739269371


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,89,10669739345434
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,63,10669739346928


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,138,10669739419263
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,148,10669739418303


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,184,10669739480307
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,110,10669739484347


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,87,10669739549086
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,49,10669739551376


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,64,10669739619610
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,62,10669739619785


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,72,10669739682903
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,142,10669739679324


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,98,10669739735951
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,138,10669739733436


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,74001.40,74001.50,0.01351324872688305431853024502,109,10669739790923
1,ETHUSDT,live,False,2029.14,2029.15,0.04928184038104718982625687174,83,10669739792401


## 7. Stop The Background Thread

Always stop the service when you are done with the notebook kernel or before re-running the start cell.

In [ ]:
service.stop(timeout=10.0)
service.status()